# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook chạy được trên Colab/Kaggle hoặc từ thư mục gốc của repo. Các lượt huấn luyện tốn thời gian được bảo vệ bằng cờ `RUN_*`; chỉ bật chúng khi đã có GPU và đã kiểm tra pipeline. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet trong Settings.
!pip -q install timm openpyxl

import sys, os, platform
from pathlib import Path
import numpy as np, pandas as pd, torch, timm

# Tự tìm thư mục repo: chạy được cả khi notebook mở trong starter/ lẫn ở thư mục cha.
_candidates = [Path.cwd(), Path.cwd() / 'K4-Track4-Day2-Deeplearning-Advance', Path.cwd().parent]
REPO_DIR = next((p.resolve() for p in _candidates if (p / 'eval.py').is_file()), None)
if REPO_DIR is None:
    raise FileNotFoundError('Không tìm thấy eval.py. Hãy clone repo và chạy notebook từ repo hoặc thư mục cha.')
CODE_DIR = REPO_DIR / ('code' if (REPO_DIR / 'code').is_dir() else 'starter')
sys.path[:0] = [str(REPO_DIR), str(CODE_DIR)]

print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("repo:", REPO_DIR)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
if not torch.cuda.is_available():
    print('Chỉ chạy EDA/pipeline trên CPU; hãy dùng GPU trước khi bật các cờ RUN_* cho thí nghiệm đầy đủ.')

### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [ ]:
import hashlib

DATA_DIR = REPO_DIR / 'data'
LABELS_DIR = DATA_DIR / 'labels'
DATA_DIR.mkdir(exist_ok=True)
LABELS_DIR.mkdir(exist_ok=True)
if not (DATA_DIR / 'images.zip').exists():
    !wget -q -O {DATA_DIR}/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open(DATA_DIR / 'images.zip', "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

!unzip -q -n {DATA_DIR}/images.zip -d {DATA_DIR}/
!ls {DATA_DIR} | head

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O {LABELS_DIR}/{name}.csv {BASE}/{name}.csv
!ls -la {LABELS_DIR}

In [ ]:
# Zenodo giải nén trực tiếp thành data/images/. Dừng sớm nếu cấu trúc khác.
IMAGES_DIR = DATA_DIR / 'images'
if not IMAGES_DIR.is_dir() or not any(IMAGES_DIR.glob('*.jpg')):
    raise FileNotFoundError(f'Không thấy ảnh JPG trong {IMAGES_DIR}; kiểm tra lại vị trí giải nén.')
print(f"images: {IMAGES_DIR} ({sum(1 for _ in IMAGES_DIR.glob('*.jpg')):,} JPG)")
print(f'labels: {LABELS_DIR}')

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import dataset

train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
split_stats = dataset.check_split(train_df, val_df, test_df, IMAGES_DIR)
labels_df = pd.read_csv(LABELS_DIR / 'labels.csv').drop_duplicates('Label').sort_values('Label')
class_names = labels_df['Species'].tolist()
display(pd.DataFrame(split_stats['per_class']).rename(index=dict(enumerate(class_names))).rename_axis('class'))
print('Số ảnh:', split_stats['n'], '| giao:', split_stats['overlap'], '| hợp:', split_stats['union'])

counts = pd.concat([
    frame.assign(split=split).groupby(['split', 'Label']).size().rename('n')
    for split, frame in [('train', train_df), ('val', val_df), ('test', test_df)]
]).reset_index()
counts['class'] = counts['Label'].map(dict(enumerate(class_names)))
pivot = counts.pivot(index='class', columns='split', values='n').reindex(class_names)
ax = pivot.plot(kind='bar', figsize=(14, 5), title='Phân bố lớp của DeepWeeds — fold 0')
ax.set(xlabel='Lớp', ylabel='Số ảnh'); plt.xticks(rotation=35, ha='right'); plt.tight_layout(); plt.show()
total_counts = pd.concat([train_df, val_df, test_df]).Label.value_counts()
print(f'Tỉ lệ lớp lớn nhất/nhỏ nhất: {total_counts.max() / total_counts.min():.2f}x')

# Mỗi lớp ba ảnh train, chỉ để EDA; không thay đổi split hay nhãn.
examples = train_df.groupby('Label', group_keys=False).sample(n=3, random_state=7)
fig, axes = plt.subplots(9, 3, figsize=(9, 24))
for label in range(dataset.NUM_CLASSES):
    rows = examples.loc[examples['Label'].eq(label)]
    for col, (_, row) in enumerate(rows.iterrows()):
        axes[label, col].imshow(Image.open(IMAGES_DIR / row.Filename).convert('RGB'))
        axes[label, col].set_title(class_names[label])
        axes[label, col].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
import torch.nn.functional as F
from torch import nn
import losses, model as model_utils
from train import set_seed

SEED = 0
set_seed(SEED)
train_loader = dataset.make_loader(train_df, IMAGES_DIR, dataset.build_transforms(True, 224, 'basic'), 4, True, num_workers=0)
images, targets, names = next(iter(train_loader))
sanity_net = model_utils.build_model('mobilenetv3', pretrained=False, num_classes=dataset.NUM_CLASSES).eval()
with torch.inference_mode():
    initial_loss = F.cross_entropy(sanity_net(images), targets).item()
print(f'Initial CE = {initial_loss:.4f}; reference -ln(1/9) = {np.log(9):.4f}')
ce = F.cross_entropy(sanity_net(images), targets)
focal0 = losses.FocalLoss(gamma=0)(sanity_net(images), targets)
assert torch.allclose(ce, focal0, atol=1e-6), 'Focal gamma=0 phải bằng CE'

# Hiển thị augmentation sau khi giải chuẩn hoá.
mean = torch.tensor(dataset.IMAGENET_MEAN).view(3, 1, 1); std = torch.tensor(dataset.IMAGENET_STD).view(3, 1, 1)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, image, target, filename in zip(axes.flat, images[:8], targets[:8], names[:8]):
    ax.imshow((image.cpu() * std + mean).permute(1, 2, 0).clamp(0, 1))
    ax.set_title(f'{class_names[int(target)]}\n{filename}')
    ax.axis('off')
plt.tight_layout(); plt.show()

# Overfit thật sự trên một batch; bật mặc định khi có GPU để không khóa notebook CPU hàng giờ.
RUN_OVERFIT_CHECK = torch.cuda.is_available()
if RUN_OVERFIT_CHECK:
    device = torch.device('cuda')
    overfit_net = model_utils.build_model('mobilenetv3', pretrained=False, num_classes=9).to(device).train()
    xb, yb = images[:8].to(device), targets[:8].to(device)
    optimizer = torch.optim.AdamW(overfit_net.parameters(), lr=3e-3, weight_decay=0)
    for step in range(250):
        optimizer.zero_grad(set_to_none=True); loss = F.cross_entropy(overfit_net(xb), yb); loss.backward(); optimizer.step()
        if loss.item() < 0.02: break
    print(f'Overfit batch: step={step + 1}, loss={loss.item():.5f}')
    assert loss.item() < 0.1, 'Chưa overfit được batch nhỏ: dừng và kiểm tra pipeline.'
else:
    print('Bỏ qua overfit check trên CPU. Chạy lại ô này trên GPU trước khi bắt đầu thí nghiệm thật.')

## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
from dataclasses import asdict
from train import Config, run

# Tất cả dùng cùng recipe T00; tag pretrained được lấy từ timm và sẽ ghi vào kết quả.
BACKBONE_PLAN = [
    ('B01', 'resnet50'), ('B02', 'resnext50'), ('B03', 'convnext_tiny'),
    ('B04', 'deit_small'), ('B05', 'mobilenetv3'),
]
BASELINE = dict(seed=SEED, fold=0, init='finetune', img_size=224, aug='basic', loss='ce',
                epochs=12, batch_size=64, lr_backbone=1e-4, lr_head=1e-3, weight_decay=0.05,
                warmup_epochs=1, amp=True, num_workers=2, images_dir=str(IMAGES_DIR),
                labels_dir=str(LABELS_DIR), out_dir=str(REPO_DIR / 'runs'), pred_dir=str(REPO_DIR / 'predictions'))
RUN_BACKBONE_SWEEP = False  # Bật trên GPU sau khi Bước 0 thành công.
backbone_records = []
if RUN_BACKBONE_SWEEP:
    if not torch.cuda.is_available(): raise RuntimeError('Không chạy sweep đầy đủ trên CPU.')
    for exp_id, backbone in BACKBONE_PLAN:
        result = run(Config(exp_id=exp_id, backbone=backbone, **BASELINE))
        cfg = Config(exp_id=exp_id, backbone=backbone, **BASELINE)
        pretrained_cfg = timm.get_pretrained_cfg(model_utils.SUGGESTED_BACKBONES.get(backbone, backbone))
        backbone_records.append({**asdict(cfg), 'pretrained_tag': getattr(pretrained_cfg, 'tag', None), **result})
backbone_table = pd.DataFrame(backbone_records)
display(backbone_table)
if len(backbone_table):
    SELECTED_BACKBONE = backbone_table.sort_values('macro_f1', ascending=False).iloc[0].backbone
    print('Backbone đi tiếp (macro-F1 val cao nhất):', SELECTED_BACKBONE)
else:
    SELECTED_BACKBONE = 'convnext_tiny'  # chỉ là giá trị chờ; sẽ tự thay sau khi sweep chạy
    print('Chưa có sweep; SELECTED_BACKBONE tạm thời =', SELECTED_BACKBONE)

## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
# Mỗi hàng chỉ đổi một yếu tố so với T00. Có bốn trục: A init, B augmentation, C loss, F EMA.
TRAINING_PLAN = [
    ('T00', 'baseline', {}),
    ('T01', 'A: khởi tạo', {'init': 'scratch'}),
    ('T02', 'A: khởi tạo', {'init': 'frozen'}),
    ('T03', 'B: augmentation', {'aug': 'color'}),
    ('T04', 'B: augmentation', {'mix': 'cutmix', 'mix_alpha': 1.0}),
    ('T05', 'C: loss', {'loss': 'ls', 'label_smoothing': 0.1}),
    ('T06', 'C: loss', {'loss': 'focal', 'focal_gamma': 2.0}),
    ('T07', 'F: regularization', {'ema_decay': 0.999}),
]
RUN_TRAINING_ABLATION = False
training_records = []
if RUN_TRAINING_ABLATION:
    if not torch.cuda.is_available(): raise RuntimeError('Không chạy ablation đầy đủ trên CPU.')
    for exp_id, axis, changed in TRAINING_PLAN:
        cfg_dict = {**BASELINE, **changed, 'exp_id': exp_id, 'backbone': SELECTED_BACKBONE}
        result = run(Config(**cfg_dict))
        training_records.append({**cfg_dict, 'axis': axis, 'changed': repr(changed), **result})
training_table = pd.DataFrame(training_records)
if len(training_table):
    baseline_f1 = training_table.loc[training_table.exp_id.eq('T00'), 'macro_f1'].iloc[0]
    training_table['delta_vs_T00'] = training_table.macro_f1 - baseline_f1
display(training_table)

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
import json
import inference, benchmark
from eval import compute_metrics

# Inference chỉ đọc checkpoint đã chọn trên val. Không đọc test ở bước này.
INFERENCE_EXP_ID, INFERENCE_SEED = 'T00', 0
checkpoint_dir = REPO_DIR / 'runs' / INFERENCE_EXP_ID / f'seed{INFERENCE_SEED}'
checkpoint_path, config_path = checkpoint_dir / 'best.pt', checkpoint_dir / 'config.json'
RUN_INFERENCE_STUDY = False  # Bật sau khi đã có checkpoint; toàn bộ so sánh dùng VAL.
inference_records = []

def add_inference_record(exp_id, method, y_true, probs, k=1):
    metrics = compute_metrics(y_true, probs.argmax(1), probs)
    inference_records.append({'exp_id': exp_id, 'method': method, 'k': k, **metrics})

if RUN_INFERENCE_STUDY:
    if not checkpoint_path.is_file(): raise FileNotFoundError(f'Chưa có checkpoint: {checkpoint_path}')
    trained_cfg = json.loads(config_path.read_text(encoding='utf-8'))
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    infer_net = model_utils.build_model(trained_cfg['backbone'], pretrained=False, num_classes=9,
                                        drop_rate=trained_cfg['drop_rate'], init=trained_cfg['init']).to(device)
    infer_net.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True)['state_dict'])
    val_loader = dataset.make_loader(val_df, IMAGES_DIR, dataset.build_transforms(False, trained_cfg['img_size']),
                                     trained_cfg['batch_size'], False, num_workers=trained_cfg['num_workers'])
    names, y_val, logits_1 = inference.predict_logits(infer_net, val_loader, device)
    _, y_flip, logits_flip = inference.predict_logits(infer_net, val_loader, device, inference.view_hflip)
    assert np.array_equal(y_val, y_flip)
    probs_1 = inference.aggregate_views([logits_1], 'prob')
    probs_flip = inference.aggregate_views([logits_1, logits_flip], 'prob')
    probs_logit = inference.aggregate_views([logits_1, logits_flip], 'logit')
    temperature = inference.fit_temperature(logits_1, y_val)
    probs_temp = inference.apply_temperature(logits_1, temperature)
    add_inference_record('I00', '1-view', y_val, probs_1)
    add_inference_record('I01', 'TTA horizontal flip, probability mean', y_val, probs_flip, k=2)
    add_inference_record('I02', 'TTA horizontal flip, logit mean', y_val, probs_logit, k=2)
    add_inference_record('I03', f'temperature scaling (T={temperature:.4f})', y_val, probs_temp)
    # Bốn phương pháp ngoài I00: thêm multi-scale (224/256) và ensemble khi có checkpoint thứ hai.
    scale256 = lambda x: torch.nn.functional.interpolate(x, size=(256, 256), mode='bilinear', align_corners=False)
    _, y_scale, logits_scale = inference.predict_logits(infer_net, val_loader, device, scale256)
    probs_scale = inference.aggregate_views([logits_1, logits_scale], 'prob')
    add_inference_record('I04', 'TTA 224/256, probability mean', y_scale, probs_scale, k=2)
    # Đo riêng từng loại chi phí: 1-view/temperature có cùng một forward; mỗi TTA phải chạy K forward thật.
    bench_kw = dict(batch_size=1, img_size=trained_cfg['img_size'], dtype='amp' if device.type == 'cuda' else 'fp32',
                    device=str(device), warmup=10, iters=50)
    one_view_latency = benchmark.latency_report(infer_net, **bench_kw)
    two_view_latency = benchmark.tta_latency(infer_net, k_views=2, **bench_kw)
    x_bench = torch.randn(1, 3, trained_cfg['img_size'], trained_cfg['img_size'], device=device)
    sync = torch.cuda.synchronize if device.type == 'cuda' else None
    def multiscale_forward():
        with torch.inference_mode(), torch.autocast(device_type=device.type, enabled=bench_kw['dtype'] == 'amp' and device.type == 'cuda'):
            infer_net(x_bench); infer_net(torch.nn.functional.interpolate(x_bench, size=(256, 256), mode='bilinear', align_corners=False))
    multiscale_latency = benchmark.bench(multiscale_forward, warmup=10, iters=50, sync=sync)
    latency_by_id = {'I00': one_view_latency, 'I01': two_view_latency, 'I02': two_view_latency,
                     'I03': one_view_latency, 'I04': multiscale_latency}
    for row in inference_records:
        row.update({f'latency_{key}_ms': latency_by_id[row['exp_id']][key] for key in ('p50', 'p95', 'p99')})
    display(pd.DataFrame(inference_records)); print('Latency by method:', latency_by_id)
else:
    print('Bỏ qua inference study: đặt RUN_INFERENCE_STUDY=True sau khi có checkpoint T00/seed0.')

## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `python eval.py score` và `python eval.py grade` (ô dưới).

In [ ]:
from eval import save_predictions

# Được xác định sau Step 1--3 bằng VAL. Nếu training_table tồn tại, chọn hàng có macro-F1 cao nhất.
_changes = {exp_id: changed for exp_id, _, changed in TRAINING_PLAN}
BEST_TRAINING_ID = (training_table.sort_values('macro_f1', ascending=False).iloc[0].exp_id
                    if len(training_table) else 'T00')
FINAL_OVERRIDES = _changes[BEST_TRAINING_ID]
FINAL_INFERENCE = 'temperature'  # Chỉ đổi sau khi so sánh inference trên VAL.
FINAL_EXP_ID, SEEDS = 'F01', (0, 1, 2)
RUN_FINAL_TEST = False  # CẢNH BÁO: bật đúng một lần, chỉ sau khi chốt bằng VAL.

if RUN_FINAL_TEST:
    if not torch.cuda.is_available(): raise RuntimeError('Chạy chung kết trên GPU, không dùng CPU.')
    final_records = []
    for seed in SEEDS:
        # Mốc T00/I00 và cấu hình cuối chạy cùng seed; train.run chỉ chạm test ở đây.
        baseline_cfg = Config(**{**BASELINE, 'exp_id': 'T00', 'seed': seed, 'backbone': SELECTED_BACKBONE,
                               'save_test_predictions': True})
        final_cfg = Config(**{**BASELINE, **FINAL_OVERRIDES, 'exp_id': FINAL_EXP_ID, 'seed': seed,
                            'backbone': SELECTED_BACKBONE, 'save_test_predictions': True})
        baseline_result, final_result = run(baseline_cfg), run(final_cfg)
        final_records.extend([{'exp_id': 'T00', 'seed': seed, **baseline_result},
                              {'exp_id': FINAL_EXP_ID, 'seed': seed, **final_result}])
        if FINAL_INFERENCE == 'temperature':
            # Giữ bản chưa hiệu chuẩn, fit T trên val cùng seed, rồi ghi đè F01 bằng dự đoán đã hiệu chuẩn.
            run_dir = REPO_DIR / 'runs' / FINAL_EXP_ID / f'seed{seed}'
            val_pred = pd.read_csv(REPO_DIR / 'predictions' / f'{FINAL_EXP_ID}_seed{seed}_val.csv')
            test_pred = pd.read_csv(REPO_DIR / 'predictions' / f'{FINAL_EXP_ID}_seed{seed}_test.csv')
            np.save(run_dir / 'test_logits.npy', np.log(test_pred[[f'p{i}' for i in range(9)]].to_numpy().clip(1e-12, 1)))
            val_logits = np.load(run_dir / 'val_logits.npy')
            T = inference.fit_temperature(val_logits, val_pred.y_true.to_numpy())
            uncal = REPO_DIR / 'predictions' / f'{FINAL_EXP_ID}uncal_seed{seed}_test.csv'
            (REPO_DIR / 'predictions' / f'{FINAL_EXP_ID}_seed{seed}_test.csv').replace(uncal)
            save_predictions(REPO_DIR / 'predictions' / f'{FINAL_EXP_ID}_seed{seed}_test.csv',
                             test_pred.Filename, test_pred.y_true, inference.apply_temperature(np.load(run_dir / 'test_logits.npy'), T))
    final_table = pd.DataFrame(final_records)
    display(final_table)
else:
    print('Chưa chạy test. Chốt backbone/training/inference trên VAL, rồi bật RUN_FINAL_TEST đúng một lần.')

In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (không sửa eval.py).
import subprocess
prediction_dir, eval_out = REPO_DIR / 'predictions', REPO_DIR / 'eval_out'
for tag in ('F01', 'T00'):
    files = sorted(prediction_dir.glob(f'{tag}_seed*_test.csv'))
    if not files:
        print(f'Chưa có predictions cho {tag}; chỉ chạy ô này sau Bước 4.')
        continue
    subprocess.run([sys.executable, str(REPO_DIR / 'eval.py'), 'score', '--pred', str(prediction_dir / f'{tag}_seed*_test.csv'),
                    '--test-csv', str(LABELS_DIR / 'test_subset0.csv'), '--labels', str(LABELS_DIR / 'labels.csv'),
                    '--tag', tag, '--out', str(eval_out)], check=True)

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận).
final_files = sorted(prediction_dir.glob('F01_seed*_test.csv'))
baseline_files = sorted(prediction_dir.glob('T00_seed*_test.csv'))
if final_files and baseline_files:
    command = [sys.executable, str(REPO_DIR / 'eval.py'), 'grade', '--final', str(prediction_dir / 'F01_seed*_test.csv'),
               '--baseline', str(prediction_dir / 'T00_seed*_test.csv'), '--test-csv', str(LABELS_DIR / 'test_subset0.csv'),
               '--labels', str(LABELS_DIR / 'labels.csv'), '--out', str(eval_out)]
    uncal_files = sorted(prediction_dir.glob('F01uncal_seed*_test.csv'))
    final_val_files = sorted(prediction_dir.glob('F01_seed*_val.csv'))
    if uncal_files: command += ['--uncal', str(prediction_dir / 'F01uncal_seed*_test.csv')]
    if final_val_files: command += ['--final-val', str(prediction_dir / 'F01_seed*_val.csv')]
    subprocess.run(command, check=True)
else:
    print('Chưa đủ prediction của F01 và T00 để chấm.')

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
import shutil

# Tập hợp các bảng đã sinh ra ở các bước trên. Không điền số thủ công: mọi số phải truy về log/eval_out.
backbone_table = globals().get('backbone_table', pd.DataFrame())
training_table = globals().get('training_table', pd.DataFrame())
inference_table = pd.DataFrame(globals().get('inference_records', []))
final_table = globals().get('final_table', pd.DataFrame())
per_class_path = eval_out / 'F01_per_class.csv'
per_class = pd.read_csv(per_class_path) if per_class_path.is_file() else pd.DataFrame()
latency_table = inference_table[[c for c in inference_table.columns if 'latency' in c or c in {'exp_id', 'method', 'k'}]].copy() if len(inference_table) else pd.DataFrame()
summary_candidates = pd.concat([
    backbone_table.assign(stage='backbone'), training_table.assign(stage='training'),
    inference_table.assign(stage='inference'), final_table.assign(stage='final')
], ignore_index=True, sort=False)
summary_table = summary_candidates.sort_values('macro_f1', ascending=False).head(10) if 'macro_f1' in summary_candidates else summary_candidates

submission_curves = REPO_DIR / 'curves'; submission_curves.mkdir(exist_ok=True)
for curve in (REPO_DIR / 'runs').glob('*/*/curves.png') if (REPO_DIR / 'runs').is_dir() else []:
    exp_id = curve.parents[1].name
    shutil.copy2(curve, submission_curves / f'{exp_id}_{curve.parent.name}.png')

xlsx_path = REPO_DIR / 'results.xlsx'
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    backbone_table.to_excel(writer, sheet_name='Backbones', index=False)
    training_table.to_excel(writer, sheet_name='Training', index=False)
    inference_table.to_excel(writer, sheet_name='Inference', index=False)
    final_table.to_excel(writer, sheet_name='Final', index=False)
    per_class.to_excel(writer, sheet_name='PerClass', index=False)
    latency_table.to_excel(writer, sheet_name='Latency', index=False)
    summary_table.to_excel(writer, sheet_name='Summary', index=False)
print('Đã ghi', xlsx_path, 'và sao chép biểu đồ vào', submission_curves)
print('Chỉ nộp results.xlsx sau khi các bảng chứa kết quả chạy thật và đã đối chiếu eval.py.')